In [1]:
import pandas as pd
import folium

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv("../data/spacex_cleaned.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset Shape: (90, 19)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class,Year
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0,2010
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0,2012
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0,2013
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0,2013
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0,2013


In [3]:
launch_sites = df[["LaunchSite", "Latitude", "Longitude"]].drop_duplicates()

print(launch_sites)

      LaunchSite   Latitude   Longitude
0   CCAFS SLC 40  28.561857  -80.577366
3    VAFB SLC 4E  34.632093 -120.610829
26    KSC LC 39A  28.608059  -80.603956


In [4]:
spacex_map = folium.Map(
    location=[29.5, -95.0],
    zoom_start=4
)

for index, row in launch_sites.iterrows():

    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        popup=row["LaunchSite"],
        tooltip=row["LaunchSite"],
        icon=folium.Icon(color="red", icon="info-sign")
    ).add_to(spacex_map)

spacex_map

In [5]:
site_analysis = df.groupby("LaunchSite").agg(
    Total_Launches=("Class", "count"),
    Successful_Landings=("Class", "sum")
).reset_index()

site_analysis["Success_Rate"] = (
    site_analysis["Successful_Landings"] /
    site_analysis["Total_Launches"] * 100
)

site_analysis

,LaunchSite,Total_Launches,Successful_Landings,Success_Rate
0,CCAFS SLC 40,55,33,60.000000
1,KSC LC 39A,22,17,77.272727
2,VAFB SLC 4E,13,10,76.923077


In [6]:
spacex_success_map = folium.Map(
    location=[29.5, -95.0],
    zoom_start=4
)

for index, row in site_analysis.iterrows():

    coordinates = df[df["LaunchSite"] == row["LaunchSite"]][
        ["Latitude", "Longitude"]
    ].iloc[0]

    popup_text = f"""
    <b>Launch Site:</b> {row['LaunchSite']}<br>
    <b>Total Launches:</b> {row['Total_Launches']}<br>
    <b>Successful Landings:</b> {row['Successful_Landings']}<br>
    <b>Success Rate:</b> {row['Success_Rate']:.2f}%
    """

    folium.Marker(
        location=[coordinates["Latitude"], coordinates["Longitude"]],
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=row["LaunchSite"],
        icon=folium.Icon(color="green", icon="rocket", prefix="fa")
    ).add_to(spacex_success_map)

spacex_success_map

In [7]:
spacex_success_map.save("../visualizations/spacex_launch_map.html")

print("Interactive map saved successfully!")

Interactive map saved successfully!
